<a href="https://colab.research.google.com/github/saurabhsen848/Generative-AI_Assignment/blob/main/Assignment_5_StyleGAN2_Latent_Space.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5: Exploring the Latent Space of a Pretrained StyleGAN2 Model

**Objective:** Understand how changes in the latent space of a GAN create smooth changes in generated images.

**Important:** This notebook uses NVIDIA's official pretrained StyleGAN2-ADA model (trained on FFHQ faces). No training is required — we only use it to generate images.

**Before running:** In Colab, go to Runtime → Change runtime type → select **GPU**. This will not run well on CPU.


## Step 1: Clone the official StyleGAN2-ADA-PyTorch repo and install requirements

In [ ]:
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch
!pip install ninja -q


## Step 2: Imports

In [ ]:
import torch
import numpy as np
import dnnlib
import legacy
import matplotlib.pyplot as plt
import imageio
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Step 3: Load the pretrained StyleGAN2 model (FFHQ faces)

This downloads NVIDIA's official pretrained weights — it may take a minute or two.


In [ ]:
network_pkl = "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl"

with dnnlib.util.open_url(network_pkl) as f:
    G = legacy.load_network_pkl(f)['G_ema'].to(device)

print("Model loaded. Latent dimension (z_dim):", G.z_dim)


## Step 4: Helper function to generate an image from a latent vector z

In [ ]:
def generate_image(z, truncation_psi=0.7):
    z = z.to(device)
    label = torch.zeros([1, G.c_dim], device=device)  # no class labels for FFHQ
    img = G(z, label, truncation_psi=truncation_psi, noise_mode="const")
    img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    return img[0].cpu().numpy()


## Step 5: Generate two images from two random latent vectors (z1 and z2)

In [ ]:
torch.manual_seed(42)  # remove or change this for different random results

z1 = torch.randn([1, G.z_dim])
z2 = torch.randn([1, G.z_dim])

img1 = generate_image(z1)
img2 = generate_image(z2)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img1)
axes[0].set_title("Image from z1")
axes[0].axis("off")
axes[1].imshow(img2)
axes[1].set_title("Image from z2")
axes[1].axis("off")
plt.show()


## Step 6: Linear interpolation between z1 and z2 (10 intermediate steps)

In [ ]:
n_steps = 10
alphas = np.linspace(0, 1, n_steps)

interpolated_images = []
for alpha in alphas:
    z_interp = (1 - alpha) * z1 + alpha * z2
    img = generate_image(z_interp)
    interpolated_images.append(img)

fig, axes = plt.subplots(1, n_steps, figsize=(20, 2.5))
for i, img in enumerate(interpolated_images):
    axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(f"{alphas[i]:.1f}", fontsize=8)
plt.suptitle("Linear Interpolation from z1 (left) to z2 (right)")
plt.tight_layout()
plt.show()


## Step 7: Save the interpolation as a GIF and as a single image strip

In [ ]:
# Save as GIF
imageio.mimsave("interpolation.gif", interpolated_images, duration=0.3)
print("Saved interpolation.gif")

# Save as a single horizontal image strip
strip = np.concatenate(interpolated_images, axis=1)
Image.fromarray(strip).save("interpolation_strip.png")
print("Saved interpolation_strip.png")

plt.figure(figsize=(20, 2.5))
plt.imshow(strip)
plt.axis("off")
plt.title("Interpolation Image Strip")
plt.show()


## Step 8: Experiment — change only one or two dimensions of a latent vector

We take one base latent vector and change just 1–2 of its dimensions to see if any specific visible feature changes (e.g., pose, smile, hair, lighting).


In [ ]:
base_z = torch.randn([1, G.z_dim])
base_img = generate_image(base_z)

# Try changing a single dimension across a range of values
dim_to_change = 5  # try changing this index to explore different dimensions
values = np.linspace(-3, 3, 6)

modified_images = []
for v in values:
    z_mod = base_z.clone()
    z_mod[0, dim_to_change] = v
    modified_images.append(generate_image(z_mod))

fig, axes = plt.subplots(1, len(values), figsize=(15, 3))
for i, img in enumerate(modified_images):
    axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(f"dim {dim_to_change} = {values[i]:.1f}", fontsize=8)
plt.suptitle(f"Effect of changing latent dimension {dim_to_change}")
plt.tight_layout()
plt.show()


In [ ]:
# Try a second dimension for comparison
dim_to_change_2 = 20  # try a different index

modified_images_2 = []
for v in values:
    z_mod = base_z.clone()
    z_mod[0, dim_to_change_2] = v
    modified_images_2.append(generate_image(z_mod))

fig, axes = plt.subplots(1, len(values), figsize=(15, 3))
for i, img in enumerate(modified_images_2):
    axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(f"dim {dim_to_change_2} = {values[i]:.1f}", fontsize=8)
plt.suptitle(f"Effect of changing latent dimension {dim_to_change_2}")
plt.tight_layout()
plt.show()
